# LegalQA v9.1 — grouped-dev candidate oracle và selector calibration
Attach source v9.1, official data, bốn approved base models, completed global dense index và QLoRA `split` artifact. Notebook dùng `selector_train.json` cho kNN nên 400 reference dev không lọt vào train-memory.

In [ ]:
TRIALS = 3000
WORKING = '/kaggle/working/legalqa_selector_calibration'


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
def is_v9_source(path):
    return (path / 'kaggle' / 'run_selector_calibration.py').is_file() and (path / 'src' / 'legalqa' / 'train_memory.py').is_file()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml') if is_v9_source(p.parent)})
if len(source_projects) != 1:
    raise RuntimeError(f'Cần đúng một source v9.1: {source_projects}')
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(source_projects[0], project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)


In [ ]:
command = [
    sys.executable, str(project / 'kaggle' / 'run_selector_calibration.py'),
    '--input-root', '/kaggle/input',
    '--working', WORKING,
    '--trials', str(TRIALS),
]
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path(WORKING)
summary = json.loads((root / 'calibration_summary.json').read_text(encoding='utf-8'))
assert 300 <= summary['grouped_dev_count'] <= 500
assert (root / 'tuned_selection.yaml').is_file()
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Lưu tuned_selection.yaml thành Kaggle Dataset và attach vào notebook public v9.')
